In [14]:
import httpx
import time
import pandas as pd
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import json
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F

In [15]:
language_classifier = pipeline(
    "text-classification",
    model = "papluca/xlm-roberta-base-language-detection"
)

def lang_result(text):
    results = language_classifier(
        text,
        truncation=True
    )
    return results[0]["label"]


model_name = "yangheng/deberta-v3-base-absa-v1.1"

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()


def aspect_sentiment(text, aspect, batch_size=16, max_length=512, stride=64):
    encoded = tokenizer(
        text,
        aspect,
        truncation=True,
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        padding=True,
        return_tensors="pt"
    )

    input_keys = ["input_ids", "attention_mask", "token_type_ids"]
    input_keys = [k for k in input_keys if k in encoded]

    all_probs = []

    with torch.inference_mode():
        n_chunks = encoded["input_ids"].shape[0]

        for start in range(0, n_chunks, batch_size):
            end = start + batch_size

            batch = {
                k: encoded[k][start:end].to(device)
                for k in input_keys
            }

            outputs = model(**batch)
            probs = F.softmax(outputs.logits, dim=-1)
            all_probs.append(probs)

    avg_probs = torch.cat(all_probs, dim=0).mean(dim=0).cpu()

    return {
        model.config.id2label[i]: float(avg_probs[i])
        for i in range(len(avg_probs))
    }

Device set to use mps:0


Using device: mps


/Users/mnatali/Projects/sentiment_analysis/.venv/lib/python3.13/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [16]:
print(aspect_sentiment("I love the weather here! However, datacenters are a concern. The friendly neighbors hake up for it though!", "datacenters"))
sentiment = aspect_sentiment("I love the weather here! However, datacenters are a concern. The friendly neighbors hake up for it though!", "datacenters")

post_sentiments = []
post_sentiment_degrees = []

post_sentiments.append(max(sentiment, key=sentiment.get))
post_sentiment_degrees.append(max(sentiment.values()))

print(post_sentiments[0], post_sentiment_degrees[0])

{'Negative': 0.9916238188743591, 'Neutral': 0.007077292539179325, 'Positive': 0.0012989335227757692}
Negative 0.9916238188743591


In [17]:
file_path_1 = Path("/Users/mnatali/Projects/sentiment_analysis/reddit_across_subreddits_analysis/brightdata_subreddit_exports/1st_4_subreddits_description.json")
file_path_2 = Path("/Users/mnatali/Projects/sentiment_analysis/reddit_across_subreddits_analysis/brightdata_subreddit_exports/1st_4_subreddits_title.json")
file_path_3 = Path("/Users/mnatali/Projects/sentiment_analysis/reddit_across_subreddits_analysis/brightdata_subreddit_exports/2nd_4_subreddits_description.json")
file_path_4 = Path("/Users/mnatali/Projects/sentiment_analysis/reddit_across_subreddits_analysis/brightdata_subreddit_exports/2nd_4_subreddits_title.json")
file_path_5 = Path("/Users/mnatali/Projects/sentiment_analysis/reddit_across_subreddits_analysis/brightdata_subreddit_exports/3rd_4_subreddits_description.json")
file_path_6 = Path("/Users/mnatali/Projects/sentiment_analysis/reddit_across_subreddits_analysis/brightdata_subreddit_exports/3rd_4_subreddits_title.json")
file_path_7 = Path("/Users/mnatali/Projects/sentiment_analysis/reddit_across_subreddits_analysis/brightdata_subreddit_exports/4th_4_subreddits_description.json")
file_path_8 = Path("/Users/mnatali/Projects/sentiment_analysis/reddit_across_subreddits_analysis/brightdata_subreddit_exports/4th_4_subreddits_title.json")
file_path_9 = Path("/Users/mnatali/Projects/sentiment_analysis/reddit_across_subreddits_analysis/brightdata_subreddit_exports/5th_3_subreddits_description.json")
file_path_10 = Path("/Users/mnatali/Projects/sentiment_analysis/reddit_across_subreddits_analysis/brightdata_subreddit_exports/5th_3_subreddits_title.json")


with file_path_1.open("r", encoding="utf-8") as f:
    posts_1 = json.load(f)
with file_path_2.open("r", encoding="utf-8") as f:
    posts_2 = json.load(f)
with file_path_3.open("r", encoding="utf-8") as f:
    posts_3 = json.load(f)
with file_path_4.open("r", encoding="utf-8") as f:
    posts_4 = json.load(f)
with file_path_5.open("r", encoding="utf-8") as f:  
    posts_5 = json.load(f)
with file_path_6.open("r", encoding="utf-8") as f:
    posts_6 = json.load(f)
with file_path_7.open("r", encoding="utf-8") as f:
    posts_7 = json.load(f)
with file_path_8.open("r", encoding="utf-8") as f:
    posts_8 = json.load(f)
with file_path_9.open("r", encoding="utf-8") as f:
    posts_9 = json.load(f)
with file_path_10.open("r", encoding="utf-8") as f:
    posts_10 = json.load(f)

In [18]:
post_lists = [posts_1, posts_2, posts_3, posts_4, posts_5, posts_6, posts_7, posts_8, posts_9, posts_10]
subreddit_posts = []
seen_post_ids = set()

for post_list in post_lists:
    for post in post_list:
        post_id = post.get("post_id")
        if post_id in seen_post_ids:
            continue
        seen_post_ids.add(post_id)
        subreddit_posts.append(post)

print("Total posts in designated subreddits:", len(subreddit_posts))


Total posts in designated subreddits: 613


In [ ]:
'''
file_path = Path("/Users/mnatali/Projects/sentiment_analysis/brightdata_across_social_media_analysis/brightdata_social_exports/reddit_updated_datacenters_posts.json")
with file_path.open("r", encoding="utf-8") as f:
    reddit_posts = json.load(f)

nova_posts = []
virginia_posts = []
washingtondc_posts = []
arizona_posts = []
chandler_az_posts = []
texas_posts = []
dallas_posts = []
austin_posts = []
oregon_posts = []
washington_posts = []
portland_posts = []
seattle_posts = []
seattle_wa_posts = []
ohio_posts = []
columbus_posts = []
chicago_posts = []
illinois_posts = []
newyork_posts = []
newyorkcity_posts = []

for post in reddit_posts:
    if post["community_name"] != None:
        nova_posts.append(post) if post['community_name'].lower() == 'nova' else None
        virginia_posts.append(post) if post['community_name'].lower() == 'virginia' else None
        washingtondc_posts.append(post) if post['community_name'].lower() == 'washingtondc' else None
        arizona_posts.append(post) if post['community_name'].lower() == 'arizona' else None
        chandler_az_posts.append(post) if post['community_name'].lower() == 'chandleraz' else None
        texas_posts.append(post) if post['community_name'].lower() == 'texas' else None
        dallas_posts.append(post) if post['community_name'].lower() == 'dallas' else None
        austin_posts.append(post) if post['community_name'].lower() == 'austin' else None
        oregon_posts.append(post) if post['community_name'].lower() == 'oregon' else None
        washington_posts.append(post) if post['community_name'].lower() == 'washington' else None
        portland_posts.append(post) if post['community_name'].lower() == 'portland' else None
        seattle_posts.append(post) if post['community_name'].lower() == 'seattle' else None
        seattle_wa_posts.append(post) if post['community_name'].lower() == 'seattlewa' else None
        ohio_posts.append(post) if post['community_name'].lower() == 'ohio' else None
        columbus_posts.append(post) if post['community_name'].lower() == 'columbus' else None
        chicago_posts.append(post) if post['community_name'].lower() == 'chicago' else None
        illinois_posts.append(post) if post['community_name'].lower() == 'illinois' else None
        newyork_posts.append(post) if post['community_name'].lower() == 'newyork' else None
        newyorkcity_posts.append(post) if post['community_name'].lower() == 'newyorkcity' else None

subreddit_posts = [nova_posts, virginia_posts, washingtondc_posts, arizona_posts, chandler_az_posts, texas_posts, dallas_posts, austin_posts, oregon_posts, washington_posts, portland_posts, seattle_posts, seattle_wa_posts, ohio_posts, columbus_posts, chicago_posts, illinois_posts, newyork_posts, newyorkcity_posts]

print("Total posts in designated subreddits:", sum(len(sub) for sub in subreddit_posts))
'''

Total posts in designated subreddits: 360


In [ ]:
def remove_links(text):
    url_pattern = re.compile(r'[\[(]?(?:https?://|www\.)\S+[\])]?' )
    return url_pattern.sub('', text)

In [20]:
english_post_ids = []
a = 0

for sub_post in subreddit_posts:
    unclean_text = sub_post["title"] + " " + sub_post["description"] if sub_post["description"] else sub_post["title"]
    text = remove_links(unclean_text)
    language = lang_result(text)
    pid = sub_post["post_id"]
    if language == 'en':
        english_post_ids.append(pid)
    a += 1
    print("Posts scanned:", a, end="\r")

In [ ]:
post_sentiments = []
post_sentiment_degrees = []

a = 0
for post in subreddit_posts:
    a += 1
    unclean_text = post["title"] + " " + post["description"] if post["description"] else post["title"]
    text = remove_links(unclean_text)
    sentiment = aspect_sentiment(text, "datacenters")
    post_sentiments.append(max(sentiment, key=sentiment.get))
    post_sentiment_degrees.append(max(sentiment.values()))
    print("Posts scanned:", a, end="\r")

The board’s land use policy committee heard a presentation from the Department of Planning and Development about potential enhancements to data center use standards, including requiring a noise study and establishing a minimum distance from residential areas. The board had directed staff last May to provide research, findings and recommendations...  McKay said this process is designed to steer data centers into places that are most appropriate and make sure standards are set for environmental protections, noise mitigation and other issues that have people concerned. He said the county is learning from Loudoun and Prince William Counties, where data centers are becoming more and more prevalent...    A group of residents from the [Save Bren Mar From Data Center] group from the Mason District held up signs throughout Tuesday's legislative policy committee meeting, calling calling on the Fairfax county Supervisors to end by right data center development on commercial and industrial propert

In [22]:
sentiments_backup = post_sentiments.copy()
sentiment_degrees_backup = post_sentiment_degrees.copy()

data = {
    "sentiments": post_sentiments,
    "sentiment_degrees": post_sentiment_degrees
}

with open("reddit_sentiments_backup.json", "w") as f:
    json.dump(data, f)

In [23]:
posts = pd.DataFrame(columns=["ids", "text", "date", "upvotes", "number of comments", "subreddit", "sentiment", "degree", "AWS", "Amazon", "Google", "Microsoft", "Azure", "Meta", "Oracle", "Equinix", "Digital Realty", "IBM", "Facebook", "Apple", "QTS", "Vantage", "CyrusOne", "CoreSite"])

post_ids = []
post_texts = []
post_dates = []
post_num_comments = []
post_upvotes = []
post_subreddits = []

for post in subreddit_posts:
    post_ids.append(post["post_id"])
    unclean_text = post["title"] + " " + post["description"] if post["description"] else post["title"]
    text = remove_links(unclean_text)
    post_texts.append(text)
    post_dates.append(post["date_posted"])
    post_num_comments.append(post["num_comments"])
    post_upvotes.append(post["num_upvotes"])
    post_subreddits.append(post["community_name"])


posts["ids"] = post_ids
posts["text"] = post_texts
posts["date"] = post_dates
posts["number of comments"] = post_num_comments
posts["upvotes"] = post_upvotes
posts["subreddit"] = post_subreddits
posts["sentiment"] = post_sentiments
posts["degree"] = post_sentiment_degrees

In [24]:
posts = posts.astype({
    "ids": "string",
    "text": "string",
    "date": "string",
    "upvotes": "int64",
    "number of comments": "int64",
    "subreddit": "string",
    "sentiment": "string",
    "degree": "float64",
    "AWS": "bool",
    "Amazon": "bool",
    "Google": "bool",
    "Microsoft": "bool",
    "Azure": "bool",
    "Meta": "bool",
    "Oracle": "bool",
    "Equinix": "bool",
    "Digital Realty": "bool",
    "IBM": "bool",
    "Facebook": "bool",
    "Apple": "bool",
    "QTS": "bool",
    "Vantage": "bool",
    "CyrusOne": "bool",
    "CoreSite": "bool"
})

In [25]:
company_terms = ["AWS","Amazon","Google","Microsoft","Azure","Meta","Oracle","Equinix","Digital Realty","IBM","Facebook","Apple","QTS","Vantage","CyrusOne","CoreSite"]

for c in company_terms:
    posts[c] = posts["text"].str.contains(rf'\b{re.escape(c)}\b', case=False, na=False)

posts_with_company = posts[posts[company_terms].any(axis=1)]
posts_with_company.head()

,ids,text,date,upvotes,number of comments,subreddit,sentiment,degree,AWS,Amazon,...,Oracle,Equinix,Digital Realty,IBM,Facebook,Apple,QTS,Vantage,CyrusOne,CoreSite
7,t3_1u26kfu,Gov. Spanberger is wrong on data center tax br...,2026-06-10T16:07:47.344Z,358,174,Virginia,Negative,0.962869,False,True,...,False,False,False,False,False,False,False,False,False,False
9,t3_1k7ss6k,What Youngkin did for us in his last year and ...,2025-04-25T18:53:58.904Z,114,56,Virginia,Negative,0.909349,False,True,...,False,False,False,False,False,False,False,False,False,False
24,t3_1u26dts,Gov. Spanberger is wrong on data center tax br...,2026-06-10T16:01:32.558Z,277,148,nova,Negative,0.962869,False,True,...,False,False,False,False,False,False,False,False,False,False
26,t3_1sim3cp,This is what the Vantage VA2 datacenter sounds...,2026-04-11T15:29:27.904Z,1113,323,nova,Negative,0.878019,False,False,...,False,False,False,False,False,False,False,True,False,False
30,t3_1sbepfp,Video of me walking through my neighborhood ne...,2026-04-03T14:11:39.671Z,741,227,nova,Negative,0.864615,False,False,...,False,False,False,False,False,False,False,True,False,False


In [26]:
len(posts)

613

In [27]:
print(posts.columns)

Index(['ids', 'text', 'date', 'upvotes', 'number of comments', 'subreddit',
       'sentiment', 'degree', 'AWS', 'Amazon', 'Google', 'Microsoft', 'Azure',
       'Meta', 'Oracle', 'Equinix', 'Digital Realty', 'IBM', 'Facebook',
       'Apple', 'QTS', 'Vantage', 'CyrusOne', 'CoreSite'],
      dtype='object')


In [28]:
## total average sentiment calculation

pos = posts.loc[posts["sentiment"] == "Positive", "degree"].sum()
neg = posts.loc[posts["sentiment"] == "Negative", "degree"].sum()
total = posts["degree"].sum()
print("Average sentiment for specified subreddits: ", (pos-neg)/total)

Average sentiment for specified subreddits:  -0.3432580069758267


In [30]:
pos = posts.loc[posts["sentiment"] == "Positive"]
neg = posts.loc[posts["sentiment"] == "Negative"]
neutral = posts.loc[posts["sentiment"] == "Neutral"]

print("Percent of posts that are neutral:", len(neutral)/len(posts))
print("Percent of posts that are negative:", len(neg)/len(posts))
print("Percent of posts that are positive:", len(pos)/len(posts))

really_pos = pos.loc[posts["degree"] > 0.90]
really_neg = neg.loc[posts["degree"] > 0.90]

if len(neg) != 0:
    print("Percent of negative posts that are really negative:", len(really_neg)/len(neg))
if len(pos) != 0:
    print("Percent of positive posts that are really positive:", len(really_pos)/len(pos))

Percent of posts that are neutral: 0.5089722675367048
Percent of posts that are negative: 0.4094616639477977
Percent of posts that are positive: 0.08156606851549755
Percent of negative posts that are really negative: 0.3466135458167331
Percent of positive posts that are really positive: 0.04


In [31]:
def avg_sentiment_calculation(dataset):
    pos = dataset.loc[dataset["sentiment"] == "Positive", "degree"].sum()
    neg = dataset.loc[dataset["sentiment"] == "Negative", "degree"].sum()
    total = dataset["degree"].sum()
    if total != 0:
        return (pos-neg)/total
    else:
        return 0

posts['year'] = pd.to_datetime(posts['date']).dt.year

year_datasets = {year: posts[posts['year'] == year] for year in range(2010, 2027)}

for i in range(2010, 2027):
    print(f"Number of posts from {i}: ", len(year_datasets[i]))
    if(avg_sentiment_calculation(year_datasets[i]) != 0):
        print(f"Average sentiment for subreddit posts from {i}: ", avg_sentiment_calculation(year_datasets[i]))

Number of posts from 2010:  0
Number of posts from 2011:  0
Number of posts from 2012:  0
Number of posts from 2013:  0
Number of posts from 2014:  0
Number of posts from 2015:  0
Number of posts from 2016:  0
Number of posts from 2017:  0
Number of posts from 2018:  0
Number of posts from 2019:  0
Number of posts from 2020:  0
Number of posts from 2021:  0
Number of posts from 2022:  2
Number of posts from 2023:  2
Average sentiment for subreddit posts from 2023:  -0.3814494763242802
Number of posts from 2024:  8
Average sentiment for subreddit posts from 2024:  0.07318202011013966
Number of posts from 2025:  108
Average sentiment for subreddit posts from 2025:  -0.4501950084213991
Number of posts from 2026:  493
Average sentiment for subreddit posts from 2026:  -0.32721541781965724


In [32]:
def sentiment_by_company(company1, company2=None):
    company_posts = posts[
        (posts[company1] == True) | (posts[company2] == True) if company2 else (posts[company1] == True)
    ]
    return avg_sentiment_calculation(company_posts)

print("Number of Amazon-related posts:", len(posts[posts["AWS"] == True]) + len(posts[posts["Amazon"] == True]))
print("Average sentiment of AWS-related posts:", sentiment_by_company("AWS", "Amazon"))
print("Number of Google-related posts:", len(posts[posts["Google"] == True]))
print("Average sentiment of Google-related posts:", sentiment_by_company("Google"))
print("Number of Microsoft-related posts:", len(posts[posts["Microsoft"] == True]) + len(posts[posts["Azure"] == True]))
print("Average sentiment of Microsoft-related posts:", sentiment_by_company("Microsoft", "Azure"))
print("Number of Meta-related posts:", len(posts[posts["Meta"] == True]) + len(posts[posts["Facebook"] == True]))
print("Average sentiment of Meta-related posts:", sentiment_by_company("Meta"))
print("Number of Oracle-related posts:", len(posts[posts["Oracle"] == True]))
print("Average sentiment of Oracle-related posts:", sentiment_by_company("Oracle"))
print("Number of Equinix-related posts:", len(posts[posts["Equinix"] == True]))
print("Average sentiment of Equinix-related posts:", sentiment_by_company("Equinix"))
print("Number of Digital Realty-related posts:", len(posts[posts["Digital Realty"] == True]))
print("Average sentiment of Digital Realty-related posts:", sentiment_by_company("Digital Realty"))
print("Number of IBM-related posts:", len(posts[posts["IBM"] == True]))
print("Average sentiment of IBM-related posts:", sentiment_by_company("IBM"))
print("Number of Apple-related posts:", len(posts[posts["Apple"] == True]))
print("Average sentiment of Apple-related posts:", sentiment_by_company("Apple"))
print("Number of QTS-related posts:", len(posts[posts["QTS"] == True]))
print("Average sentiment of QTS-related posts:", sentiment_by_company("QTS"))
print("Number of Vantage-related posts:", len(posts[posts["Vantage"] == True]))
print("Average sentiment of Vantage-related posts:", sentiment_by_company("Vantage"))
print("Number of CyrusOne-related posts:", len(posts[posts["CyrusOne"] == True]))
print("Average sentiment of CyrusOne-related posts:", sentiment_by_company("CyrusOne"))
print("Number of CoreSite-related posts:", len(posts[posts["CoreSite"] == True]))
print("Average sentiment of CoreSite-related posts:", sentiment_by_company("CoreSite"))

Number of Amazon-related posts: 43
Average sentiment of AWS-related posts: -0.32370245989473206
Number of Google-related posts: 29
Average sentiment of Google-related posts: -0.3323696865547027
Number of Microsoft-related posts: 14
Average sentiment of Microsoft-related posts: -0.6128122161835642
Number of Meta-related posts: 27
Average sentiment of Meta-related posts: -0.4230756178877735
Number of Oracle-related posts: 2
Average sentiment of Oracle-related posts: 0.36756329238739976
Number of Equinix-related posts: 1
Average sentiment of Equinix-related posts: -1.0
Number of Digital Realty-related posts: 0
Average sentiment of Digital Realty-related posts: 0
Number of IBM-related posts: 0
Average sentiment of IBM-related posts: 0
Number of Apple-related posts: 10
Average sentiment of Apple-related posts: -0.2917653143481629
Number of QTS-related posts: 2
Average sentiment of QTS-related posts: -1.0
Number of Vantage-related posts: 14
Average sentiment of Vantage-related posts: -0.6712

In [33]:
def analysis_of_viral_posts(min):
    viral_posts = posts.loc[posts["upvotes"] >= min]
    non_viral_posts = posts.loc[posts["upvotes"] < min]
    print("What's considered viral: posts with over", min, "upvotes")
    print("Number of viral posts:", len(viral_posts))
    print("Average sentiment of viral posts:", avg_sentiment_calculation(viral_posts))
    print("Average sentiment of non-viral posts:", avg_sentiment_calculation(non_viral_posts))
    print("Amount percent of viral posts that are polar (degree) > 0.90: ", len(viral_posts.loc[viral_posts["degree"] > 0.90])/len(viral_posts))
    print("Percent of viral posts that are negative: ", (len(viral_posts.loc[viral_posts["sentiment"] == "Negative"]))/len(viral_posts))
    print("Percent of non-viral posts that are negative: ", (len(non_viral_posts.loc[non_viral_posts["sentiment"] == "Negative"]))/(len(non_viral_posts)))

analysis_of_viral_posts(500)
print("\n")
analysis_of_viral_posts(750)
print("\n")
analysis_of_viral_posts(1000)
print("\n")
analysis_of_viral_posts(2000)

What's considered viral: posts with over 500 upvotes
Number of viral posts: 121
Average sentiment of viral posts: -0.5053991083100198
Average sentiment of non-viral posts: -0.3026995739189659
Amount percent of viral posts that are polar (degree) > 0.90:  0.4049586776859504
Percent of viral posts that are negative:  0.5371900826446281
Percent of non-viral posts that are negative:  0.3780487804878049


What's considered viral: posts with over 750 upvotes
Number of viral posts: 64
Average sentiment of viral posts: -0.45588183084149253
Average sentiment of non-viral posts: -0.32986222391217906
Amount percent of viral posts that are polar (degree) > 0.90:  0.40625
Percent of viral posts that are negative:  0.484375
Percent of non-viral posts that are negative:  0.4007285974499089


What's considered viral: posts with over 1000 upvotes
Number of viral posts: 41
Average sentiment of viral posts: -0.4643524761424108
Average sentiment of non-viral posts: -0.33428872334024107
Amount percent of v

In [34]:
nova_posts = posts.loc[posts["subreddit"].str.lower() == "nova"]
virginia_posts = posts.loc[posts["subreddit"].str.lower() == "virginia"]
washingtondc_posts = posts.loc[posts["subreddit"].str.lower() == "washingtondc"]
arizona_posts = posts.loc[posts["subreddit"].str.lower() == "arizona"]
chandleraz_posts = posts.loc[posts["subreddit"].str.lower() == "chandleraz"]
texas_posts = posts.loc[posts["subreddit"].str.lower() == "texas"]
dallas_posts = posts.loc[posts["subreddit"].str.lower() == "dallas"]
austin_posts = posts.loc[posts["subreddit"].str.lower() == "austin"]
oregon_posts = posts.loc[posts["subreddit"].str.lower() == "oregon"]
portland_posts = posts.loc[posts["subreddit"].str.lower() == "portland"]
washington_posts = posts.loc[posts["subreddit"].str.lower() == "washington"]
seattle_posts = posts.loc[posts["subreddit"].str.lower() == "seattle"]
seattlewa_posts = posts.loc[posts["subreddit"].str.lower() == "seattlewa"]
ohio_posts = posts.loc[posts["subreddit"].str.lower() == "ohio"]
columbus_posts = posts.loc[posts["subreddit"].str.lower() == "columbus"]
illinois_posts = posts.loc[posts["subreddit"].str.lower() == "illinois"]
chicago_posts = posts.loc[posts["subreddit"].str.lower() == "chicago"]
newyork_posts = posts.loc[posts["subreddit"].str.lower() == "newyork"]
newyorkcity_posts = posts.loc[posts["subreddit"].str.lower() == "newyorkcity"]

virginia = pd.concat(
    [nova_posts, virginia_posts]
)
arizona = pd.concat(
    [arizona_posts, chandleraz_posts]
)
texas = pd.concat(
    [texas_posts, dallas_posts, austin_posts]
)
oregon = pd.concat(
    [oregon_posts, portland_posts]
)
washington = pd.concat(
    [washington_posts, seattle_posts, seattlewa_posts]
)
ohio = pd.concat(
    [ohio_posts, columbus_posts]
)
illinois = pd.concat(
    [illinois_posts, chicago_posts]
)
newyork = pd.concat(
    [newyork_posts, newyorkcity_posts]
)

print("Total posts from Virginia:", len(virginia))
print("Total posts from Washington DC:", len(washingtondc_posts))
print("Total posts from Arizona:", len(arizona))
print("Total posts from Texas:", len(texas))
print("Total posts from Oregon:", len(oregon))
print("Total posts from Washington:", len(washington))
print("Total posts from Ohio:", len(ohio))
print("Total posts from Illinois:", len(illinois))
print("Total posts from New York:", len(newyork))

state_wide_posts = pd.concat(
    [virginia_posts, arizona_posts, texas_posts, oregon_posts, washington_posts, ohio_posts, newyork_posts]
)
city_or_region_posts = pd.concat(
    [nova_posts, chandleraz_posts, dallas_posts, austin_posts, portland_posts, seattle_posts, seattlewa_posts, columbus_posts, chicago_posts, newyorkcity_posts]
)

print("Total posts from state-wide subreddits:", len(state_wide_posts))
print("Total posts from city or region-based subreddits:", len(city_or_region_posts))

Total posts from Virginia: 196
Total posts from Washington DC: 3
Total posts from Arizona: 13
Total posts from Texas: 75
Total posts from Oregon: 41
Total posts from Washington: 43
Total posts from Ohio: 181
Total posts from Illinois: 54
Total posts from New York: 7
Total posts from state-wide subreddits: 339
Total posts from city or region-based subreddits: 228


In [35]:
print("Average sentiment of posts from Virginia:", avg_sentiment_calculation(virginia))
print("Average sentiment of posts from Washington DC:", avg_sentiment_calculation(washingtondc_posts))
print("Average sentiment of posts from Arizona:", avg_sentiment_calculation(arizona))
print("Average sentiment of posts from Texas:", avg_sentiment_calculation(texas))
print("Average sentiment of posts from Oregon:", avg_sentiment_calculation(oregon))
print("Average sentiment of posts from Washington:", avg_sentiment_calculation(washington))
print("Average sentiment of posts from Ohio:", avg_sentiment_calculation(ohio))
print("Average sentiment of posts from Illinois:", avg_sentiment_calculation(illinois))
print("Average sentiment of posts from New York:", avg_sentiment_calculation(newyork))
print("\n")

print("Average sentiment of state-wide subreddits:", avg_sentiment_calculation(state_wide_posts))
print("Average sentiment of city of region-based subreddits", avg_sentiment_calculation(city_or_region_posts))

Average sentiment of posts from Virginia: -0.2941655615532987
Average sentiment of posts from Washington DC: -0.43272236046111706
Average sentiment of posts from Arizona: -0.4121552847806757
Average sentiment of posts from Texas: -0.27426670145333204
Average sentiment of posts from Oregon: -0.32222196857584434
Average sentiment of posts from Washington: -0.3028059582821063
Average sentiment of posts from Ohio: -0.4429399706994147
Average sentiment of posts from Illinois: -0.3116463019675193
Average sentiment of posts from New York: -0.26917965590644183


Average sentiment of state-wide subreddits: -0.37373612765191333
Average sentiment of city of region-based subreddits -0.3112893759600236
